#  MoneyMap — Tutorial

> Conversión de divisas y cálculo de impuestos con **precisión decimal exacta**.

Este notebook instala MoneyMap directamente desde PyPI y demuestra todas sus funcionalidades paso a paso.



## 0 · Instalación

Instalamos MoneyMap con soporte para pandas y polars:

In [1]:
%pip install moneymap[pandas,polars] --quiet

Note: you may need to restart the kernel to use updated packages.


## 1 · Conversión de divisas

MoneyMap usa `Decimal` internamente para evitar errores de punto flotante comunes al trabajar con dinero.

### ¿Por qué Decimal y no float?

In [2]:
# El problema con float
print(0.1 + 0.2)           # 0.30000000000000004
print(0.1 + 0.2 == 0.3)    # False  <- peligroso en código financiero

# Con Decimal
from decimal import Decimal
print(Decimal('0.1') + Decimal('0.2'))                    # 0.3
print(Decimal('0.1') + Decimal('0.2') == Decimal('0.3'))  # True

0.30000000000000004
False
0.3
True


### Conversiones básicas

In [3]:
from moneymap import convertir

convertir(1000, "MXN", "USD")   # Decimal('58.65')

Decimal('58.65')

In [4]:
convertir(100, "USD", "MXN")    # Decimal('1705.00')

Decimal('1705.00')

In [5]:
convertir(500, "EUR", "GBP")    # Decimal('429.35')

Decimal('429.35')

In [6]:
# Acepta int, float, str o Decimal
print(convertir(100,      "USD", "MXN"))
print(convertir(100.50,   "USD", "MXN"))
print(convertir("100.50", "USD", "MXN"))

1705.00
1713.53
1713.53


## 2 · Registro de tasas custom

`registrar_tasa(divisa, referencia, tasa)` — establece cuántas unidades de `divisa` hay por 1 unidad de `referencia`.

In [7]:
from moneymap import registrar_tasa

print("Antes: ", convertir(1000, "JPY", "MXN"))  # ~114 MXN

# Establecer que 1 JPY = 0.10 MXN
registrar_tasa("MXN", "JPY", 0.10)

print("Después:", convertir(1000, "JPY", "MXN"))  # 100.00 MXN

Antes:  114.05
Después: 100.00


In [8]:
# Restaurar y agregar divisa nueva
registrar_tasa("MXN", "USD", 17.05)
registrar_tasa("VEF", "USD", 36.50)
convertir(100, "USD", "VEF")

Decimal('3650.00')

## 3 · Cálculo de impuestos

Más de 35 países incluidos: IVA, VAT, GST, etc.

In [9]:
from moneymap import impuesto
from moneymap.taxes import total_con_impuesto, tasa_fiscal

print(impuesto(1000, "Mexico"))           # 160.00  (IVA 16%)
print(impuesto(1000, "España"))           # 210.00  (IVA 21%)
print(impuesto(1000, "USA"))              # 0.00    (sin IVA federal)
print(total_con_impuesto(1000, "Mexico")) # 1160.00

160.00
210.00
0.00
1160.00


In [10]:
# Tabla comparativa
monto  = 1000
paises = ["Mexico", "Argentina", "España", "Alemania", "USA", "Suecia", "Japon"]

print(f"{'País':<20} {'Tasa':>6} {'Impuesto':>10} {'Total':>10}")
print("-" * 50)
for pais in paises:
    print(
        f"{pais:<20}"
        f" {str(tasa_fiscal(pais))+'%':>6}"
        f" {str(impuesto(monto, pais)):>10}"
        f" {str(total_con_impuesto(monto, pais)):>10}"
    )

País                   Tasa   Impuesto      Total
--------------------------------------------------
Mexico                  16%     160.00    1160.00
Argentina               21%     210.00    1210.00
España                  21%     210.00    1210.00
Alemania                19%     190.00    1190.00
USA                      0%       0.00    1000.00
Suecia                  25%     250.00    1250.00
Japon                   10%     100.00    1100.00


## 4 · Catálogos disponibles

In [11]:
from moneymap import divisas_disponibles, paises_disponibles

print(f"Divisas ({len(divisas_disponibles())}): {divisas_disponibles()}")
print()
print(f"Países ({len(paises_disponibles())}): {paises_disponibles()}")

Divisas (16): ['ARS', 'AUD', 'BRL', 'CAD', 'CHF', 'CLP', 'CNY', 'COP', 'EUR', 'GBP', 'INR', 'JPY', 'MXN', 'PEN', 'USD', 'VEF']

Países (44): ['Alemania', 'Argentina', 'Australia', 'Austria', 'Belgica', 'Bolivia', 'Brasil', 'Canada', 'Chile', 'China', 'Colombia', 'Corea del Sur', 'Costa Rica', 'Cuba', 'Dinamarca', 'Ecuador', 'El Salvador', 'España', 'Finlandia', 'Francia', 'Grecia', 'Guatemala', 'Honduras', 'India', 'Italia', 'Japon', 'Mexico', 'Nicaragua', 'Noruega', 'Nueva Zelanda', 'Paises Bajos', 'Panama', 'Paraguay', 'Peru', 'Polonia', 'Portugal', 'Reino Unido', 'Republica Dominicana', 'Singapur', 'Suecia', 'Suiza', 'USA', 'Uruguay', 'Venezuela']


## 5 · Manejo de errores

In [12]:
from moneymap.exceptions import (
    DivisaNoSoportadaError, PaisNoSoportadoError,
    MontoInvalidoError, TasaInvalidaError,
)

casos = [
    ("Divisa inválida",   lambda: convertir(100, "XYZ", "USD")),
    ("País no existe",    lambda: impuesto(100, "Neverland")),
    ("Monto negativo",    lambda: convertir(-50, "USD", "MXN")),
    ("Monto no numérico", lambda: convertir("abc", "USD", "MXN")),
    ("Tasa inválida",     lambda: registrar_tasa("TST", "USD", -5)),
]

for label, fn in casos:
    try:
        fn()
    except (DivisaNoSoportadaError, PaisNoSoportadoError,
            MontoInvalidoError, TasaInvalidaError) as e:
        print(f"⚠ {label}: {e}")

⚠ Divisa inválida: La divisa 'XYZ' no está soportada. Usa divisas_disponibles() para ver las opciones.
⚠ País no existe: El país 'Neverland' no tiene reglas fiscales registradas. Usa paises_disponibles() para ver las opciones.
⚠ Monto negativo: El monto '-50' no es válido. El monto no puede ser negativo.
⚠ Monto no numérico: El monto 'abc' no es válido. No es un número válido.
⚠ Tasa inválida: La tasa de cambio '-5' es inválida. La tasa debe ser mayor que cero.


## 6 · Ayuda integrada

`ayuda()` funciona como `man` en bash.

In [13]:
from moneymap import ayuda
ayuda()              # índice general


══════════════════════════════════════════════════════════
  MoneyMap — Referencia de funciones
══════════════════════════════════════════════════════════

  Uso: ayuda('nombre_funcion')  para ver detalles.

  Divisas
    convertir(monto, origen, destino)
      Convierte un monto de una divisa a otra con precisión decimal exacta.
    registrar_tasa(divisa, referencia, tasa)
      Registra o actualiza la tasa de cambio de una divisa.
    divisas_disponibles()
      Devuelve la lista de códigos de divisas soportadas, ordenada alfabéticamente.

  Impuestos
    impuesto(monto, pais)
      Calcula el monto del impuesto (IVA/VAT/GST) según el país.
    total_con_impuesto(monto, pais)
      Calcula el monto total (base + impuesto) para un país.
    tasa_fiscal(pais)
      Devuelve el porcentaje de impuesto registrado para un país.
    paises_disponibles()
      Devuelve la lista de países con reglas fiscales registradas, ordenada alfabéticamente.

  Importar desde el paquete principal:
    f

In [14]:
ayuda("convertir")   # detalle completo


──────────────────────────────────────────────────────────
  convertir(monto, origen, destino)
──────────────────────────────────────────────────────────

  Convierte un monto de una divisa a otra con precisión decimal exacta.

  Argumentos
    monto  int | float | str | Decimal
      Valor a convertir. No puede ser negativo.
    origen  str
      Código ISO de la divisa de origen (ej. 'MXN').
    destino  str
      Código ISO de la divisa de destino (ej. 'USD').

  Retorna
    Decimal — resultado redondeado a 2 decimales.

  Lanza
    MontoInvalidoError  —  Si el monto no es numérico o es negativo.
    DivisaNoSoportadaError  —  Si alguna divisa no está registrada.

  Ejemplo
    from moneymap import convertir
    
    convertir(1000, "MXN", "USD")   # → Decimal('58.65')
    convertir(100,  "USD", "MXN")   # → Decimal('1705.00')
    convertir(500,  "EUR", "GBP")   # → Decimal('429.35')

──────────────────────────────────────────────────────────



## 7 · Integración con pandas

Se activa con un solo import. Todas las operaciones son no destructivas y encadenables.

In [ ]:
import pandas as pd
import moneymap.dataframepd   # registra df.moneymap

df = pd.DataFrame({
    "producto": ["Laptop", "Monitor", "Teclado", "Mouse", "Webcam"],
    "precio":   [25000,    8500,      1200,      350,     950],
})
df

,producto,precio
0,Laptop,25000
1,Monitor,8500
2,Teclado,1200
3,Mouse,350
4,Webcam,950


In [16]:
df.moneymap.convertir(col="precio", origen="MXN", destino="USD")

,producto,precio,precio_USD
0,Laptop,25000,1466.28
1,Monitor,8500,498.53
2,Teclado,1200,70.38
3,Mouse,350,20.53
4,Webcam,950,55.72


In [17]:
df.moneymap.resumen_fiscal(col="precio", pais="España")

,producto,precio,precio_impuesto,precio_total,precio_tasa_pct
0,Laptop,25000,5250.0,30250.0,21.0
1,Monitor,8500,1785.0,10285.0,21.0
2,Teclado,1200,252.0,1452.0,21.0
3,Mouse,350,73.5,423.5,21.0
4,Webcam,950,199.5,1149.5,21.0


In [18]:
# Encadenado
(
    df
    .moneymap.convertir(col="precio", origen="MXN", destino="USD")
    .moneymap.impuesto(col="precio_USD", pais="USA")
    .moneymap.total_con_impuesto(col="precio_USD", pais="USA")
)

,producto,precio,precio_USD,precio_USD_impuesto,precio_USD_total
0,Laptop,25000,1466.28,0.0,1466.28
1,Monitor,8500,498.53,0.0,498.53
2,Teclado,1200,70.38,0.0,70.38
3,Mouse,350,20.53,0.0,20.53
4,Webcam,950,55.72,0.0,55.72


## 8 · Integración con Polars

Mismo API para `DataFrame` y `LazyFrame`. Las funciones `expr_*` devuelven `pl.Expr` para `.with_columns()`.

In [ ]:
import polars as pl
import moneymap.dataframepl as mm

df_pl = pl.DataFrame({
    "producto": ["Laptop", "Monitor", "Teclado", "Mouse", "Webcam"],
    "precio":   [25000,    8500,      1200,      350,     950],
})

mm.convertir(df_pl, col="precio", origen="MXN", destino="USD")

ImportError: attempted relative import with no known parent package

In [ ]:
# LazyFrame
(
    df_pl.lazy()
    .pipe(mm.convertir, col="precio", origen="MXN", destino="USD")
    .pipe(mm.impuesto,  col="precio_USD", pais="USA")
    .collect()
)

In [ ]:
# Expresiones — múltiples columnas en una sola pasada
df_pl.with_columns(
    mm.expr_convertir("precio", "MXN", "USD").alias("precio_USD"),
    mm.expr_convertir("precio", "MXN", "EUR").alias("precio_EUR"),
    mm.expr_impuesto("precio", "Mexico").alias("iva"),
    mm.expr_total_con_impuesto("precio", "Mexico").alias("precio_total"),
)

## 9 · Caso de uso real

Tabla de ventas internacionales: convertimos todos los montos a USD y calculamos el impuesto local de cada país.

In [ ]:
ventas = pd.DataFrame({
    "orden":    [1001,      1002,    1003,     1004,          1005],
    "producto": ["Laptop",  "Curso", "Plugin", "Libro",       "Suscripción"],
    "monto":    [1200,      49,      19,        25,            9.99],
    "divisa":   ["USD",     "EUR",   "USD",     "GBP",         "USD"],
    "pais":     ["Mexico",  "España", "USA",    "Reino Unido", "Argentina"],
})

# Convertir cada monto a USD según su divisa de origen
ventas["monto_usd"] = ventas.apply(
    lambda r: float(convertir(r["monto"], r["divisa"], "USD")), axis=1
)

# Impuesto local de cada país
ventas["impuesto_usd"] = ventas.apply(
    lambda r: float(impuesto(r["monto_usd"], r["pais"])), axis=1
)

# Total con impuesto
ventas["total_usd"] = ventas.apply(
    lambda r: float(total_con_impuesto(r["monto_usd"], r["pais"])), axis=1
)

ventas

In [ ]:
print(f"Subtotal (USD):          ${ventas['monto_usd'].sum():.2f}")
print(f"Total impuestos (USD):   ${ventas['impuesto_usd'].sum():.2f}")
print(f"Total con impuestos:     ${ventas['total_usd'].sum():.2f}")